# Review 2: full-row preparation and training-throughput benchmark

Import this notebook into Kaggle; enable **Internet** and **Tesla T4 GPU**; attach the private, reviewed `cic-collection.parquet`. Run top to bottom. This prepares the entire combined flow collection into row-aligned disk arrays, but benchmarks training on **at most 100,000 training rows** only. It does not train a full-data classifier, inspect test metrics, or run poisoning/continual experiments. Per-row source provenance is unverified. The original split is row-level and cannot guarantee independent duplicate flows or capture sessions.

For a restart in the same Kaggle working directory, set `RESUME_OUTPUT_DIR` below to the printed folder. Completed stages have manifest markers; interrupted stages are safely rebuilt. Persist the full `/kaggle/working` output as a Kaggle notebook version or private Kaggle dataset before ending the session; the prepared arrays are too large for Git.


## 1. Clone the exact published source and inspect the runtime

The pinned source contains the disk-backed preparation and training APIs. Do not install local requirements over Kaggle's CUDA-enabled PyTorch.


In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib, shutil, platform, resource, os
from dataclasses import asdict
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '5c0d519596bc6c4b99948126443635c2f7eb15f1'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-large-prepare')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import pyarrow as pa
import torch
from src.data import inspect_parquet, BROAD_CLASS_ORDER
from src.data.large_data import large_data_paths, prepare_full_partitions, prepare_raw_feature_store, fit_disk_preprocessor, verify_full_partitions
from src.training.disk_backed import DiskBackedFlowDataset, fit_disk_backed
from src.models import ModelConfig, TabularTransformerClassifier
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required: enable Kaggle GPU and restart')
environment = {'python': platform.python_version(), 'packages': {name: version(name) for name in ('numpy', 'pyarrow', 'torch')}, 'torch_cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0)}
print('Verified source SHA:', checked_out_sha, 'environment:', environment)
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_large_data.py', '-v'], cwd=REPO_DIR, check=True, capture_output=True, text=True)
print(result.stdout, end=''); print(result.stderr, end='')
import re
match = re.search(r'Ran (\d+) tests? in', result.stdout + result.stderr)
if match is None or int(match.group(1)) == 0 or 'skipped' in (result.stdout + result.stderr).split('OK')[-1]:
    raise RuntimeError('Large-data tests were not fully discovered and run on Kaggle')


## 2. Identify the exact input, checksum it, and reserve output space

Edit `PARQUET_PATH` only if discovery finds zero or multiple candidates. A mismatched checksum stops the run. The estimate includes one float32 feature store, row/split arrays, explicit split indices, temporary reservoir memory, and headroom. Hashing itself scans the full input. No output directory is created until these checks pass.


In [ ]:
PARQUET_PATH = None  # editable: '/kaggle/input/.../cic-collection.parquet'
EXPECTED_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
EXPECTED_ROWS = 9_167_581
RESUME_OUTPUT_DIR = None  # editable: printed /kaggle/working/review2_large_prepare_... path
candidates = sorted(Path('/kaggle/input').rglob('cic-collection.parquet'))
print('Input candidates:', [str(path) for path in candidates])
if PARQUET_PATH is None:
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one attached input; found {len(candidates)}. Set PARQUET_PATH.')
    parquet_path = candidates[0]
else:
    parquet_path = Path(PARQUET_PATH)
    if not parquet_path.is_file() or not parquet_path.is_relative_to(Path('/kaggle/input')):
        raise ValueError('PARQUET_PATH must be an attached Kaggle input file')
digest = hashlib.sha256(); start = perf_counter()
with parquet_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
input_sha256 = digest.hexdigest(); checksum_seconds = perf_counter() - start
if input_sha256 != EXPECTED_SHA256:
    raise RuntimeError('Input checksum differs from reviewed combined flow collection')
info = inspect_parquet(parquet_path)
assert info.row_count == EXPECTED_ROWS and len(info.schema.retained_feature_names) == 54
storage_bytes_estimate = info.row_count * (54 * 4 + 8 + 1 + 1 + 8)  # features, IDs, labels, splits, explicit indices
required_free_bytes = int(storage_bytes_estimate * 1.5 + 1_000_000_000)
available_bytes = shutil.disk_usage('/kaggle/working').free
print('Checksum:', input_sha256, 'rows:', info.row_count, 'features:', len(info.schema.retained_feature_names))
if RESUME_OUTPUT_DIR is None:
    OUTPUT_DIR = Path('/kaggle/working') / ('review2_large_prepare_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
else:
    OUTPUT_DIR = Path(RESUME_OUTPUT_DIR)
    if not OUTPUT_DIR.is_dir() or not OUTPUT_DIR.is_relative_to(Path('/kaggle/working')):
        raise ValueError('RESUME_OUTPUT_DIR must be an existing /kaggle/working preparation folder')
remaining_required = 1_000_000_000 if (OUTPUT_DIR / 'features_manifest.json').exists() else required_free_bytes
print('Estimated persisted bytes:', storage_bytes_estimate, 'required free for remaining stages:', remaining_required, 'available:', available_bytes)
if available_bytes < remaining_required:
    raise RuntimeError('Insufficient /kaggle/working space for safe remaining preparation')
OUTPUT_DIR.mkdir(exist_ok=True)
manifest_path = OUTPUT_DIR / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    if manifest['source_sha'] != checked_out_sha or manifest['input_sha256'] != input_sha256:
        raise RuntimeError('Resume folder belongs to a different source or input')
else:
    manifest = {'purpose': 'full-row preparation and training-only throughput benchmark, not full training', 'source_sha': checked_out_sha, 'input_sha256': input_sha256, 'input_bytes': parquet_path.stat().st_size, 'input_rows': info.row_count, 'environment': environment, 'provenance': 'combined flow collection; per-row source provenance unverified', 'split_seed': 42, 'preprocessing_reservoir_size': 200_000, 'preprocessing_reservoir_seed': 42, 'timings_seconds': {'checksum': checksum_seconds}, 'stages': {}}
    manifest_path.write_text(json.dumps(manifest, indent=2))
print('Preparation folder:', OUTPUT_DIR)


## 3. Persist and verify the complete 70/15/15 stratified split

Scan only `ClassLabel`; use stable global IDs 0–7 and seed 42. Write original row IDs, label IDs, split codes, and explicit split-index files. Verification checks every original position exactly once, no overlapping assignments, and each class's partition counts. This is a row-level split: duplicate flows or capture-session correlation may cross partitions.


In [ ]:
def progress(message):
    print(message, flush=True)
start = perf_counter()
split_summary = prepare_full_partitions(parquet_path, OUTPUT_DIR, seed=42, progress=progress)
split_seconds = perf_counter() - start
assert split_summary == verify_full_partitions(large_data_paths(OUTPUT_DIR), info.row_count) | {'seed': 42, 'fractions': [0.70, 0.15, 0.15], 'split_codes': {'train': 0, 'validation': 1, 'test': 2}}
manifest['stages']['split'] = {'completed': True, 'counts': split_summary['counts'], 'seconds_this_invocation': split_seconds}
manifest_path.write_text(json.dumps(manifest, indent=2))
print('Split counts:', split_summary['counts'])


## 4. Build one bounded-batch, row-aligned raw feature store

Read only the 54 retained numeric features plus `ClassLabel` for an alignment check. `Label` and `ClassLabel` are **never features**. Convert each batch to float32 on disk and mark negative/non-finite values missing, following the existing baseline. No full raw float64 matrix or repeated full-array concatenation is created. The completed stage can be reused on restart.


In [ ]:
start = perf_counter()
feature_path = prepare_raw_feature_store(parquet_path, OUTPUT_DIR, progress=progress)
feature_seconds = perf_counter() - start
feature_store = np.load(feature_path, mmap_mode='r', allow_pickle=False)
assert feature_store.shape == (EXPECTED_ROWS, 54) and feature_store.dtype == np.float32
feature_meta = json.loads((OUTPUT_DIR / 'features_manifest.json').read_text())
assert feature_meta['targets_excluded'] == ['Label', 'ClassLabel']
manifest['stages']['feature_store'] = {'completed': True, 'seconds_this_invocation': feature_seconds, 'file_bytes': feature_path.stat().st_size, 'feature_names': feature_meta['feature_names']}
manifest_path.write_text(json.dumps(manifest, indent=2))
print('Feature store:', feature_path, 'bytes:', feature_path.stat().st_size)


## 5. Fit two separate training-only preprocessing states

Static state uses all full-training rows; continual state uses only Task 1 training IDs 0–4, then stays frozen for future Task 2 rows. Both choose up to 200,000 seeded training rows **without replacement** to approximate per-feature medians. After imputation, a bounded pass computes population mean/std over the entire authorized fit scope. This approximation and its seed/size are saved; validation/test never fit either state. The original small-array exact-median API is unchanged.


In [ ]:
states = {}
for scope in ('static', 'continual'):
    start = perf_counter()
    states[scope] = fit_disk_preprocessor(OUTPUT_DIR, scope=scope, reservoir_size=200_000, seed=42, progress=progress)
    elapsed = perf_counter() - start
    record = json.loads((OUTPUT_DIR / f'preprocessing_{scope}.json').read_text())
    assert record['scope'] == scope and record['reservoir_size_used'] <= 200_000
    assert np.isfinite(states[scope].imputation_values).all() and np.isfinite(states[scope].means).all() and np.isfinite(states[scope].scales).all()
    manifest['stages'][f'preprocessing_{scope}'] = {'completed': True, 'fit_rows': record['fit_rows'], 'reservoir_size_used': record['reservoir_size_used'], 'seconds_this_invocation': elapsed}
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(scope, 'fit rows:', record['fit_rows'], 'reservoir:', record['reservoir_size_used'], 'seconds:', elapsed)


## 6. Fix one naturally distributed training-only benchmark subset

Select up to 100,000 original IDs uniformly from the full **training partition** with seed 42. Save those IDs; do not include validation/test rows or balance classes artificially. The disk view loads/transforms only requested batches. This subset is for throughput only, not classification or strategy selection.


In [ ]:
all_train_indices = np.load(OUTPUT_DIR / 'train_indices.npy', mmap_mode='r', allow_pickle=False)
BENCHMARK_LIMIT = 100_000
BENCHMARK_SEED = 42
benchmark_index_path = OUTPUT_DIR / 'benchmark_train_indices.npy'
if benchmark_index_path.exists():
    benchmark_ids = np.load(benchmark_index_path, allow_pickle=False)
else:
    benchmark_ids = np.sort(np.random.default_rng(BENCHMARK_SEED).choice(all_train_indices, size=min(BENCHMARK_LIMIT, len(all_train_indices)), replace=False))
    np.save(benchmark_index_path, benchmark_ids)
assert benchmark_ids.size > 0 and np.all(np.load(OUTPUT_DIR / 'splits.npy', mmap_mode='r')[benchmark_ids] == 0)
benchmark_view = DiskBackedFlowDataset(OUTPUT_DIR, states['static'], partition='train', class_ids=tuple(range(8)), row_ids=benchmark_ids)
benchmark_counts = {BROAD_CLASS_ORDER[i]: int(np.count_nonzero(benchmark_view.labels[benchmark_ids] == i)) for i in range(8)}
manifest['benchmark_subset'] = {'seed': BENCHMARK_SEED, 'rows': len(benchmark_view), 'class_counts': benchmark_counts, 'index_file': benchmark_index_path.name, 'purpose': 'training-only throughput; no validation or test metrics'}
manifest_path.write_text(json.dumps(manifest, indent=2))
print('Training-only benchmark rows:', len(benchmark_view), 'natural class counts:', benchmark_counts)


## 7. Measure one training epoch at predeclared batch sizes 128 and 256

For each batch size, initialize a fresh hidden-64/two-layer Transformer with the existing model API and run one training-only epoch through the disk-backed interface. Keep CUDA mixed precision disabled initially. Synchronize CUDA around timing and report actual draws/optimizer steps, process peak RAM and peak GPU allocation. The first run may include CUDA/kernel warm-up; figures are a planning benchmark, not full training or test accuracy. The full-epoch estimate below adds a stated 1.5× headroom factor and is **not measured**. No test partition is read here.


In [ ]:
benchmark_path = OUTPUT_DIR / 'benchmark_results.json'
if benchmark_path.exists():
    benchmark = json.loads(benchmark_path.read_text())
    assert benchmark['source_sha'] == checked_out_sha and benchmark['benchmark_rows'] == len(benchmark_view)
else:
    benchmark = {'source_sha': checked_out_sha, 'benchmark_rows': len(benchmark_view), 'seed': 42, 'batch_sizes_predeclared': [128, 256], 'scope': 'static train only', 'results': {}}
for batch_size in (128, 256):
    key = str(batch_size)
    if key in benchmark['results']:
        print('Reusing completed benchmark:', key); continue
    config = ModelConfig(num_features=54, num_classes=8, hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=batch_size, learning_rate=0.001, epochs=1, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42)
    model = TabularTransformerClassifier(config, class_ids=tuple(range(8)))
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    ram_before_kib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    start = perf_counter()
    history = fit_disk_backed(model, benchmark_view, max_epochs=1, verbose=True)
    torch.cuda.synchronize(); seconds = perf_counter() - start
    actual_rows = history.training_draws
    assert actual_rows == len(benchmark_view) and history.epochs_completed == 1
    full_train_rows = sum(part['train'] for part in split_summary['counts'].values())
    rows_per_second = actual_rows / seconds
    record = {'configuration': asdict(config), 'actual_training_rows': actual_rows, 'optimizer_steps': history.optimizer_steps, 'observed_seconds': seconds, 'observed_rows_per_second': rows_per_second, 'train_loss_functional_only': history.train_loss[0], 'process_peak_ram_kib_before': ram_before_kib, 'process_peak_ram_kib_after': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, 'gpu_peak_allocated_bytes': torch.cuda.max_memory_allocated(), 'gpu_peak_reserved_bytes': torch.cuda.max_memory_reserved(), 'full_train_rows': full_train_rows, 'estimated_full_epoch_seconds_with_1_5x_headroom': 1.5 * full_train_rows / rows_per_second, 'estimate_note': 'planning estimate from subset throughput, not observed full-epoch time'}
    benchmark['results'][key] = record
    benchmark_path.write_text(json.dumps(benchmark, indent=2))
    print('batch', key, 'throughput rows/s:', rows_per_second, 'estimated full epoch seconds:', record['estimated_full_epoch_seconds_with_1_5x_headroom'])
    del model
manifest['stages']['benchmark'] = {'completed': True, 'file': benchmark_path.name}
manifest_path.write_text(json.dumps(manifest, indent=2))


## 8. Record storage costs and a provisional full-run configuration

Recommend the faster measured batch size only if both fit GPU memory; this is a throughput recommendation, not a test-tuned model choice. Future static training must use the static state; future Task 1/Task 2 training must use the frozen continual state. No full training begins in this notebook. Preserve the entire output folder in Kaggle Outputs or a private Kaggle Dataset; the next session should attach it read-only and verify its manifest/input checksum before reuse. Do not commit these arrays or benchmarks to Git.


In [ ]:
measured = benchmark['results']
recommended_batch = max((128, 256), key=lambda size: measured[str(size)]['observed_rows_per_second'])
recommended = {'status': 'provisional throughput-based recommendation, not full training', 'source_sha': checked_out_sha, 'input_sha256': input_sha256, 'model': {'hidden_dim': 64, 'num_heads': 4, 'num_layers': 2, 'mlp_dim': 128, 'batch_size': recommended_batch, 'learning_rate': 0.001, 'max_epochs': 20, 'early_stopping_patience': 5, 'mixed_precision': False, 'seed': 42}, 'static_preprocessor': 'preprocessing_static.json', 'continual_preprocessor': 'preprocessing_continual.json', 'warning': 'Full-run schedule and GPU fit must be verified in the next phase; no test tuning'}
(OUTPUT_DIR / 'recommended_full_run.json').write_text(json.dumps(recommended, indent=2))
artifact_sizes = {path.name: path.stat().st_size for path in OUTPUT_DIR.iterdir() if path.is_file()}
manifest['artifact_bytes'] = artifact_sizes
manifest['total_persisted_bytes'] = sum(artifact_sizes.values())
manifest['recommended_config_file'] = 'recommended_full_run.json'
manifest['completed_preparation_and_benchmark_only'] = True
manifest_path.write_text(json.dumps(manifest, indent=2))
print('Observed benchmark results:', json.dumps(benchmark['results'], indent=2))
print('Recommended provisional batch:', recommended_batch, 'prepared bytes:', manifest['total_persisted_bytes'])
print('Save this full folder for next session:', OUTPUT_DIR)
print('Send manifest.json, split_manifest.json, both preprocessing JSON files, benchmark_results.json, recommended_full_run.json, and this executed notebook for review. Preserve all .npy arrays in Kaggle outputs/private dataset.')
